# Price

### Packages

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox


### Google Drive

In [ ]:
drive.mount("/content/drive")

### Reading data from Google Drive

In [ ]:
# path = "/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed_mega/eurusd_master_returns_201901_202604_avg100_stride100.parquet"
# df = pd.read_parquet(path)

### Reading ask and bid data from local and using mid-price

In [ ]:
import re
from pathlib import Path

import pandas as pd

SYMBOL = "eurusd"
PROCESSED = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed")
START_YYYYMM = "202601"


def discover_bid_months() -> list[str]:
    pat = re.compile(rf"^{re.escape(SYMBOL)}_dukascopy_bid_(\d{{6}})\.parquet$", re.I)
    months: set[str] = set()
    for fp in PROCESSED.glob(f"{SYMBOL}_dukascopy_bid_*.parquet"):
        m = pat.match(fp.name)
        if m:
            months.add(m.group(1))
    return sorted(months)


def load_bid_stream(months: list[str]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for ym in months:
        fp = PROCESSED / f"{SYMBOL}_dukascopy_bid_{ym}.parquet"
        if not fp.is_file():
            continue
        d = pd.read_parquet(fp, columns=["datetime", "price"])
        d["datetime"] = pd.to_datetime(d["datetime"], utc=True)
        d = d.rename(columns={"price": "bid"})
        frames.append(d)

    if not frames:
        raise FileNotFoundError(
            f"No bid parquet files under {PROCESSED.resolve()} "
            f"(pattern {SYMBOL}_dukascopy_bid_YYYYMM.parquet)."
        )

    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values("datetime").drop_duplicates("datetime", keep="last")
    return out.reset_index(drop=True)


months = discover_bid_months()
months = [m for m in months if m >= START_YYYYMM]

df = load_bid_stream(months)
df = df.rename(columns={"bid": "price"})  # optional: keep column name "bid" instead

# optional: calendar month tag for filtering / plots
df["yyyymm"] = df["datetime"].dt.strftime("%Y%m")

### Trying to get familiar with how the prices look

In [ ]:
df.info(memory_usage="deep")
print(f"shape (rows, cols): {df.shape}")

display(df.head())
display(df.tail())
print(df.dtypes)

### Parse time

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")

tmin, tmax = df["datetime"].min(), df["datetime"].max()
print("time range:", tmin, "->", tmax)
print(df.isna().sum())
print("rows with NaN price:", df["price"].isna().sum())
print("inf price:", np.isinf(df["price"]).sum())

### Duplicates and reordering

In [ ]:
df = df.sort_values("datetime").reset_index(drop=True)

print("duplicate datetimes:", df["datetime"].duplicated().sum())
print("strictly increasing by time:", df["datetime"].is_monotonic_increasing)

### Gaps between observations

In [ ]:
dt = df["datetime"].diff().dropna()

print("Δt (seconds) — describe:\n")
print(dt.dt.total_seconds().describe(percentiles=[0.01, 0.5, 0.99, 0.999]))

worst = dt.nlargest(20)
print(worst)

### Observations per calendar year

In [ ]:
df["date"] = df["datetime"].dt.strftime("%Y-%m-%d")

daily_n = df.groupby("date", sort=True).size().rename("n_obs").reset_index()
print(daily_n["n_obs"].describe())
display(daily_n.sort_values("n_obs").head(15))
display(daily_n.sort_values("n_obs").tail(15))

### Prices and returns

In [ ]:
df = df.sort_values("datetime").reset_index(drop=True)
df["logret"] = np.log(df["price"] / df["price"].shift(1))
df["logret"] = df["logret"].replace([np.inf, -np.inf], np.nan)
print(f"nan logret: {df["logret"].isna().sum()}")

df = df.dropna(subset=["logret"]).reset_index(drop=True)
print(df[["datetime","price","logret"]].describe())

### Price plotting

In [ ]:
px = df.set_index("datetime")["price"].resample("1min").last().dropna()
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(px.index, px.values, lw=0.4, color="black")
ax.set_title("EUR/USD bid — 1-minute last")
ax.set_ylabel("price")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Histogram

In [ ]:
r = df["logret"]
# Optional: r = r.sample(min(len(r), 500_000), random_state=0)

fig, ax = plt.subplots(figsize=(12, 3))
counts, bin_edges, patches = ax.hist(r, density=True, alpha=0.75, color="steelblue", bins=1000)
ax.set_yscale("log")
ax.set_title("Histogram of tick log returns (bid)")
ax.set_xlabel("log return")
ax.set_ylabel("density (log scale)")
plt.tight_layout()
plt.show()

print(r.describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]))

### QQ-plot

In [ ]:
r = df["logret"]
# Optional: r = r.sample(min(len(r), 200_000), random_state=0)

(osm, osr), (slope, intercept, _) = stats.probplot(r, dist="norm", fit=True)

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(osm, osr, s=1, c="k", alpha=0.25, linewidths=0, rasterized=True)

xs = np.array([osm.min(), osm.max()], dtype=float)
ax.plot(xs, slope * xs + intercept, color="0.35", lw=0.8)

ax.set_xlabel("Theoretical quantiles")
ax.set_ylabel("Ordered values")
ax.set_title("QQ-plot vs normal (bid log returns)")
ax.set_aspect("auto")
plt.tight_layout()
plt.show()

### Realized variance and mean

In [ ]:
daily = (
    df.groupby("date", sort=True)["logret"]
    .agg(rv=lambda x: float(np.sum(x**2)), mabs=lambda x: float(np.mean(np.abs(x))), n="count")
    .reset_index()
)
daily["date"] = pd.to_datetime(daily["date"])
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
axes[0].plot(daily["date"], daily["rv"], lw=0.7, color="navy")
axes[0].set_ylabel("sum r²")
axes[0].set_title("Daily realized variance (bid log returns)")
axes[1].plot(daily["date"], daily["mabs"], lw=0.7, color="darkred")
axes[1].set_ylabel("mean |r|")
axes[1].set_title("Daily mean absolute log return")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
display(daily.describe())

In [ ]:
p = df.sort_values("datetime").set_index("datetime")["price"]
p = p.resample("1s").last().dropna()
r_1s = np.log(p / p.shift(1)).replace([np.inf, -np.inf], np.nan).dropna()
r_1s.name = "logret_1s"
# ACF/PACF: use a short recent slice so statsmodels stays fast
lookback = pd.Timedelta(days=5)
r_1s = r_1s.loc[r_1s.index.max() - lookback:]
print(r_1s.describe())
print("n:", len(r_1s))

In [ ]:
lags = 120
fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(
    r_1s,
    lags=lags,
    ax=ax,
    alpha=0.05,
    bartlett_confint=True,
    zero=False,
    fft=True,
)
ax.set_title("ACF — 1s log returns (bid), recent window")
plt.tight_layout()
plt.show()

In [ ]:
lags = 120
fig, ax = plt.subplots(figsize=(12, 3.5))
plot_pacf(r_1s, lags=lags, ax=ax, alpha=0.05, method="ywm", zero=False)
ax.set_title("PACF — 1s log returns (bid), recent window")
plt.tight_layout()
plt.show()

In [ ]:
lags = 120
r2 = (r_1s**2).rename("r2")
abr = r_1s.abs().rename("abs_r")
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
plot_acf(r2, lags=lags, ax=axes[0], alpha=0.05, bartlett_confint=True, zero=False, fft=True)
axes[0].set_title("ACF — (1s log return)²")
plot_acf(abr, lags=lags, ax=axes[1], alpha=0.05, bartlett_confint=True, zero=False, fft=True)
axes[1].set_title("ACF — |1s log return|")
plt.tight_layout()
plt.show()

In [ ]:
p = df.sort_values("datetime").set_index("datetime")["price"]
p = p.resample("1s").last().dropna()
r = np.log(p / p.shift(1)).replace([np.inf, -np.inf], np.nan).dropna()
r_tail = r.loc[r.index.max() - pd.Timedelta(days=14):]
x = r_tail.shift(1)
y = r_tail
m = x.notna() & y.notna()
x, y = x[m], y[m]
n = min(200_000, len(x))
rng = np.random.default_rng(0)
idx = rng.choice(len(x), size=n, replace=False)
fig, ax = plt.subplots(figsize=(6, 5))
hb = ax.hexbin(x.iloc[idx], y.iloc[idx], gridsize=80, bins="log", cmap="Greys", mincnt=1)
ax.axhline(0, color="0.5", lw=0.6)
ax.axvline(0, color="0.5", lw=0.6)
ax.set_xlabel("r_{t-1}")
ax.set_ylabel("r_t")
ax.set_title("Hexbin: 1s log return vs lag-1 (14d tail, subsampled)")
plt.colorbar(hb, ax=ax, label="log10(count)")
plt.tight_layout()
plt.show()

In [ ]:
p = df.sort_values("datetime").set_index("datetime")["price"]
p = p.resample("1s").last().dropna()
r = np.log(p / p.shift(1)).replace([np.inf, -np.inf], np.nan).dropna()
r_tail = r.loc[r.index.max() - pd.Timedelta(days=30):]
mod = r_tail.index.hour * 60 + r_tail.index.minute
s = pd.Series(np.abs(r_tail.values), index=mod).groupby(level=0).mean()
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(s.index, s.values, lw=0.8, color="black")
ax.set_xlabel("minute of day (UTC)")
ax.set_ylabel("mean |1s log ret|")
ax.set_title("Intraday seasonality (30d tail)")
ax.set_xlim(0, 24 * 60 - 1)
plt.tight_layout()
plt.show()

In [ ]:
p = df.sort_values("datetime").set_index("datetime")["price"]
p = p.resample("1s").last().dropna()
r = np.log(p / p.shift(1)).replace([np.inf, -np.inf], np.nan).dropna()
r_tail = r.loc[r.index.max() - pd.Timedelta(days=10):]
lb = acorr_ljungbox(r_tail**2, lags=[10, 20, 30], model_df=0, return_df=True)
print(lb)

In [ ]:
# pip install arch
import numpy as np
import pandas as pd
from arch import arch_model

p = df.sort_values("datetime").set_index("datetime")["price"]
p = p.resample("1s").last().dropna()
r = np.log(p / p.shift(1)).replace([np.inf, -np.inf], np.nan).dropna()

r_fit = r.loc[r.index.max() - pd.Timedelta(days=20):].dropna()
y = (r_fit * 100.0).astype(float)  # scale

am = arch_model(y, mean="Constant", vol="GARCH", p=1, q=1, rescale=False)
res = am.fit(disp="off")
print(res.summary())

In [ ]:
import matplotlib.pyplot as plt

scale = 100.0
cond_vol = res.conditional_volatility / scale
abs_r = np.abs(r_fit)

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(r_fit.index, abs_r.values, lw=0.15, color="black", alpha=0.35, label="|r|")
ax.plot(r_fit.index, cond_vol.values, lw=0.7, color="darkred", alpha=0.9, label="GARCH σ̂")
ax.set_title("GARCH(1,1) conditional vol vs |1s log return|")
ax.legend(loc="upper right")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

z = pd.Series(res.std_resid, index=r_fit.index).dropna()
z = z.loc[z.index.max() - pd.Timedelta(days=5):]  # keep ACF fast

lags = 120
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
plot_acf(z, lags=lags, ax=axes[0], alpha=0.05, bartlett_confint=True, zero=False, fft=True)
axes[0].set_title("ACF — GARCH standardized residuals")
plot_acf(z**2, lags=lags, ax=axes[1], alpha=0.05, bartlett_confint=True, zero=False, fft=True)
axes[1].set_title("ACF — squared standardized residuals")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

z = np.asarray(res.std_resid)
z = z[np.isfinite(z)]
# Optional: z = np.random.default_rng(0).choice(z, size=min(len(z), 100_000), replace=False)

(osm, osr), (slope, intercept, _) = stats.probplot(z, dist="norm", fit=True)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(osm, osr, s=2, c="k", alpha=0.2, linewidths=0, rasterized=True)
xs = np.array([osm.min(), osm.max()], dtype=float)
ax.plot(xs, slope * xs + intercept, color="0.35", lw=0.9)
ax.set_title("QQ — GARCH standardized residuals vs normal")
ax.set_xlabel("theoretical quantiles")
ax.set_ylabel("ordered values")
plt.tight_layout()
plt.show()

In [ ]:
# pip install hmmlearn
import numpy as np
import pandas as pd
from hmmlearn import hmm

p = df.sort_values("datetime").set_index("datetime")["price"]
p = p.resample("1s").last().dropna()
r = np.log(p / p.shift(1)).replace([np.inf, -np.inf], np.nan).dropna()

r_tail = r.loc[r.index.max() - pd.Timedelta(days=14):].dropna()
X = r_tail.to_numpy().reshape(-1, 1)

n_use = min(400_000, len(X))
X_fit = X[:n_use]

model = hmm.GaussianHMM(
    n_components=3,
    covariance_type="diag",
    n_iter=200,
    random_state=0,
    init_params="stmc",
)
model.fit(X_fit)
states = model.predict(X_fit)

print("startprob:\n", model.startprob_)
print("transmat:\n", model.transmat_)
print("means:\n", model.means_.ravel())

In [ ]:
import matplotlib.pyplot as plt

idx = r_tail.index[:n_use]

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(idx, X_fit.ravel(), lw=0.12, color="black", alpha=0.35)
axes[0].set_title("1s log returns (HMM training slice)")
axes[1].step(idx, states, where="post", lw=0.6)
axes[1].set_ylabel("state")
axes[1].set_title("Decoded HMM state (Viterbi)")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(model.transmat_, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xlabel("to state")
ax.set_ylabel("from state")
ax.set_title("HMM transition matrix")
for (i, j), v in np.ndenumerate(model.transmat_):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v > 0.5 else "black", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()